# 实验四：意图驱动的多模态融合优化与 AI Agent 闭环干预

## Experiment 4: Intent-Driven Multimodal Fusion with Contrastive Alignment & Agent Closed-Loop

### 问题诊断（来自实验三）

| 问题 | 实验三表现 | 本实验改进策略 |
|------|-----------|---------------|
| **模态竞争失衡** | Text gate >0.8, ECG gate <0.05 → 文本霸权 | Softmax 门控 + 门控熵正则化 + 30%文本特征掩码 |
| **伪对齐/特征稀释** | 模态特征在不同子空间，分类正确但未真正对齐 | CLIP-style InfoNCE 对比对齐损失 |
| **ECG 分支被忽略** | ECG attention 接近 0，几乎不贡献预测 | 非对称学习率(ECG 3x) + 分阶段训练(前5轮ECG+Image only) |
| **类别不平衡** | 正样本仅 8.7%，模型全阴性预测 | Focal Loss + WeightedRandomSampler |
| **AI Agent 浅参与** | 仅生成报告，不参与训练决策 | Agent 闭环干预(每3轮) + 临床先验驱动超参 + 意图注意力向量 |

### 核心创新（方法学贡献）
1. **对比对齐损失 (Contrastive Alignment Loss)**：CLIP-style InfoNCE 强制跨模态特征真对齐
2. **Softmax 门控 + 熵正则化**：替代 Sigmoid，sum=1 约束 + 最大化熵鼓励均衡
3. **分阶段训练 (Staged Training)**：前 N 轮仅 ECG+Image，打破文本优势建立 ECG 表征
4. **意图注意力向量 (Intent Attention Vector)**：Agent 从临床先验生成注意力初始化
5. **Agent 闭环干预**：每 3 轮 Agent 分析训练轨迹，动态调整超参数（JSON 解析 + 应用）
6. **文本特征掩码 (Text Masking)**：训练中随机 mask 30% 文本特征，迫使模型学习 ECG/Image

In [ ]:
# ====================================
# Cell 1: Environment & Dependencies
# ====================================
import os, sys, time, json, copy, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import wfdb
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             roc_curve, auc, confusion_matrix, precision_recall_curve,
                             classification_report, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# --- CUDA Verification ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print("Experiment 4: Improved Training Pipeline")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:      {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Using device:    {device}")

# --- AI Agent ---
from openai import OpenAI
API_KEY = "REDACTED_CONFIGURE_LOCALLY"
BASE_URL = "https://yunwu.ai/v1"
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def agent_call(system_prompt, user_prompt, model="gpt-4o-mini", max_tokens=1500):
    """Unified AI Agent call with error handling."""
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=max_tokens
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Agent Offline] {e}"

# Quick connectivity test
test_resp = agent_call("Reply OK.", "Ping", max_tokens=5)
print(f"\nAI Agent Status: {'ONLINE ✓' if 'OK' in str(test_resp).upper() or len(test_resp) < 20 else 'OFFLINE ✗'}")
print("Environment Ready.")

In [ ]:
# ====================================
# Cell 2: Load master_index.csv & Data Audit via AI Agent
# ====================================
BASE = r"e:\平时作业\人工智能\data"
MASTER_CSV = os.path.join(BASE, "master_index.csv")

df_master = pd.read_csv(MASTER_CSV)
print(f"Master Index loaded: {len(df_master)} samples")
print(f"Label distribution: {dict(df_master['label'].value_counts())}")

# --- 计算详细的不平衡统计 ---
n_pos = int(df_master['label'].sum())
n_neg = len(df_master) - n_pos
imbalance_ratio = n_neg / max(n_pos, 1)
pos_rate = n_pos / len(df_master) * 100

modality_stats = {
    'total': len(df_master),
    'positive': n_pos,
    'negative': n_neg,
    'imbalance_ratio': f'{imbalance_ratio:.1f}:1',
    'pos_rate': f'{pos_rate:.1f}%',
    'has_text': int(df_master['has_text'].sum()),
    'has_ecg': int(df_master['has_ecg'].sum()),
    'has_image': int(df_master['has_image'].sum()),
    'data_sources': dict(df_master['data_source'].value_counts())
}

print(f"\n--- Data Imbalance Summary ---")
print(f"Positive (High Risk): {n_pos} ({pos_rate:.1f}%)")
print(f"Negative (Low Risk):  {n_neg} ({100-pos_rate:.1f}%)")
print(f"Imbalance Ratio:      {imbalance_ratio:.1f}:1")

# ========== AI Agent: Data Quality Audit ==========
print("\n" + "="*60)
print("🤖 AI Agent: Data Quality Audit Report")
print("="*60)

audit_system = """You are a senior biomedical data scientist. Analyze the dataset statistics and provide:
1. **Data Quality Assessment**: Comment on imbalance severity, modality coverage, and risks.
2. **Sampling Strategy Recommendation**: Specific oversampling ratio or technique.
3. **Loss Function Recommendation**: Which loss function and its hyperparameters.
4. **Training Risk Warnings**: What could go wrong and how to mitigate.
Format your response with clear headers and bullet points. Be quantitative."""

audit_prompt = f"""Dataset Statistics:
{json.dumps(modality_stats, indent=2, default=str)}

Context: Binary classification for 30-day heart failure readmission prediction.
Previous experiment showed: model predicted all-negative (F1=0), val loss divergence at epoch 3.
Available modalities: Clinical Text (ClinicalBERT), ECG signal (1D time series), Chest X-Ray (ResNet)."""

audit_report = agent_call(audit_system, audit_prompt)
print(audit_report)

## 跨模态数据配对原理与数据增强策略

### 1. 跨模态配对的科学依据

本研究面临的核心挑战在于：**Text（临床笔记）、ECG（心电信号）和 Image（胸部X光）来自不同的患者队列**，
不存在天然的 `patient_id` 级别一一对应关系。我们采用 **标签一致性约束下的确定性轮转配对（Label-Consistent Deterministic Round-Robin Pairing）** 策略，
其科学合理性基于以下三个层面的论证：

#### 1.1 临床语义一致性假设（Clinical Semantic Coherence）

在心衰风险预测任务中，三种模态的标签含义在语义层面上具有一致性：

| 模态 | 标签=1 的语义 | 生理基础 |
|------|-------------|----------|
| **Text** (MIMIC-III/IV) | 30天内再入院 或 院内死亡 | 病程严重、合并症多、药物反应差 |
| **ECG** (PhysioNet/CPSC/CHF) | 房颤(AF) 或 充血性心衰(CHF) | 心脏电传导异常、心肌重构 |
| **Image** (CheXchonet) | 左室扩大(LVIDd↑) 或 心室肥厚(SLV/DLV) | 心脏结构性病变 |

三种标签虽然来自不同测量维度，但共同指向同一个临床终点：**心脏功能失代偿**。
将同标签的样本配对，等价于构造一个虚拟的"最坏情况"患者——该患者同时具有文本记录中的不良预后、
心电图异常信号和影像学结构异常，这符合临床上心衰高危患者的典型表现模式 (Rajkomar et al., 2018; Johnson et al., 2016)。

#### 1.2 多模态学习中的弱对齐范式（Weakly-Aligned Multimodal Learning）

在多模态学习文献中，当模态间不存在实例级配对时，**标签对齐（Label Alignment）** 是被广泛接受的弱监督策略：

$$\mathcal{D}_{aligned} = \{(x_t^{(i)}, x_e^{(j)}, x_v^{(k)}, y) \mid y_t^{(i)} = y_e^{(j)} = y_v^{(k)} = y\}$$

其中 $x_t, x_e, x_v$ 分别为文本、ECG和影像特征，$(i,j,k)$ 为不同患者的索引，$y$ 为共享标签。
这种配对方式迫使模型学习 **标签条件下的模态不变特征（Label-Conditional Modality-Invariant Features）**，
而非依赖特定患者的模态间相关性，从而提升模型的泛化能力 (Baltrušaitis et al., 2019)。

#### 1.3 确定性轮转分配的统计优势

相比随机配对（`random.choice`），确定性轮转分配（Round-Robin）具有以下优势：

- **完全可复现**：相同输入数据 → 相同配对结果，消除随机种子依赖
- **均匀覆盖**：每个 ECG/Image 样本被等频率使用，避免热点样本过拟合
- **标签一致率最大化**：优先从同标签池中取样，仅在池耗尽时才跨标签回退

### 2. 正样本过采样策略

针对正样本仅占 8.7% 的严重不平衡问题，我们采用 **PyTorch WeightedRandomSampler** 实现训练时动态过采样：

$$w_i = \begin{cases} \frac{N}{2 \cdot N_{pos}} & \text{if } y_i = 1 \\ \frac{N}{2 \cdot N_{neg}} & \text{if } y_i = 0 \end{cases}$$

其中 $N$ 为总样本数。这使得每个 epoch 中正负样本的 **期望采样频率接近 1:1**，
同时保留了负样本的多样性（不丢弃任何数据）。

### 3. Focal Loss 聚焦难分类样本

在采样层面平衡的基础上，我们在损失函数层面进一步引入 Focal Loss (Lin et al., 2017)：

$$\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

其中 $\gamma = 2$ 控制难易样本的聚焦程度，$\alpha = 0.75$ 给予正样本更高权重。
$(1-p_t)^\gamma$ 项使模型自动降低已被正确分类样本的梯度贡献，集中学习"模棱两可"的边界样本。

### 4. 跨模态数据增强

对正样本应用模态内增强以增加多样性：
- **ECG**：随机时间偏移（±200 steps）+ 高斯噪声注入（σ=0.05）
- **Image**：随机水平翻转 + 色彩抖动 + 随机仿射变换
- **Text**：同义词替换概率 p=0.1（通过 AI Agent 生成变体）

In [ ]:
# ====================================
# Cell 3: Enhanced Dataset with Oversampling & Augmentation
# ====================================

class EnhancedMultimodalDataset(Dataset):
    """Enhanced dataset with modality-specific augmentation for positive samples."""
    def __init__(self, df, transform=None, augment=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.augment = augment  # Only augment during training
        
        # Augmented transform for positive samples (stronger augmentation)
        self.aug_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        is_positive = int(row['label']) == 1
        
        # --- A. ECG with augmentation ---
        target_len = 2000
        ecg_data = np.zeros(target_len, dtype=np.float32)
        if row['has_ecg'] and pd.notna(row['ecg_path']):
            try:
                sig, meta = wfdb.rdsamp(row['ecg_path'], channels=[0])
                if sig is not None and len(sig) > 0:
                    raw = sig.flatten()
                    if np.std(raw) > 1e-4 and not np.isnan(raw).any():
                        raw = (raw - np.mean(raw)) / (np.std(raw) + 1e-6)
                        if len(raw) >= target_len:
                            start = np.random.randint(0, max(1, len(raw) - target_len))
                            ecg_data = raw[start:start + target_len].astype(np.float32)
                        else:
                            ecg_data[:len(raw)] = raw.astype(np.float32)
                        
                        # Augmentation for positive samples during training
                        if self.augment and is_positive:
                            # Random time shift (±200 steps)
                            shift = np.random.randint(-200, 200)
                            ecg_data = np.roll(ecg_data, shift)
                            # Gaussian noise injection
                            ecg_data += np.random.normal(0, 0.05, ecg_data.shape).astype(np.float32)
                            # Random amplitude scaling
                            scale = np.random.uniform(0.9, 1.1)
                            ecg_data *= scale
            except:
                pass
        ecg_tensor = torch.tensor(ecg_data, dtype=torch.float32).unsqueeze(0)
        
        # --- B. Image with augmentation ---
        try:
            if row['has_image'] and pd.notna(row['image_path']):
                image = Image.open(row['image_path']).convert('RGB')
                if self.augment and is_positive and self.aug_transform:
                    image = self.aug_transform(image)
                elif self.transform:
                    image = self.transform(image)
                else:
                    image = torch.zeros(3, 224, 224)
            else:
                image = torch.zeros(3, 224, 224)
        except:
            image = torch.zeros(3, 224, 224)
        
        # --- C. Text ---
        text_raw = ""
        if row['has_text'] and pd.notna(row['text_path']):
            try:
                with open(row['text_path'], 'r', encoding='utf-8') as f:
                    text_raw = f.read()[:2000]
            except:
                text_raw = "No clinical notes available."
        else:
            text_raw = "No clinical notes available."
        
        return {
            "ecg": ecg_tensor,
            "image": image,
            "text_raw": text_raw,
            "label": torch.tensor(float(row['label']), dtype=torch.float32),
            "data_source": str(row.get('data_source', 'Unknown')),
            "has_ecg": bool(row['has_ecg']),
            "has_image": bool(row['has_image']),
            "has_text": bool(row['has_text'])
        }

# ---- Build Datasets ----
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Stratified split (80/20) - same seed as Exp 3 for comparability
np.random.seed(42)
indices = np.arange(len(df_master))
labels_all = df_master['label'].values

# Stratified split
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=labels_all)

df_train = df_master.iloc[train_idx].reset_index(drop=True)
df_test = df_master.iloc[test_idx].reset_index(drop=True)

train_dataset = EnhancedMultimodalDataset(df_train, transform=transform_eval, augment=True)
test_dataset = EnhancedMultimodalDataset(df_test, transform=transform_eval, augment=False)

# ---- WeightedRandomSampler for class balance ----
train_labels = df_train['label'].values
n_pos_train = int(train_labels.sum())
n_neg_train = len(train_labels) - n_pos_train

# Weight per sample: inverse class frequency
weight_pos = len(train_labels) / (2.0 * max(n_pos_train, 1))
weight_neg = len(train_labels) / (2.0 * max(n_neg_train, 1))
sample_weights = np.where(train_labels == 1, weight_pos, weight_neg)
sample_weights = torch.DoubleTensor(sample_weights)

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"\n--- Enhanced Dataset Summary ---")
print(f"Train: {len(train_dataset)} (Pos: {n_pos_train}, Neg: {n_neg_train})")
print(f"Test:  {len(test_dataset)}")
print(f"Sample weights: Pos={weight_pos:.2f}, Neg={weight_neg:.2f}")
print(f"Expected per-epoch pos/neg ratio: ~1:1 (via WeightedRandomSampler)")
print(f"Augmentation: ECG(shift+noise), Image(flip+affine+jitter) for positive samples")

In [ ]:
# ====================================
# Cell 4: AI Agent — Hyperparameter Recommendation
# ====================================
print("="*60)
print("🤖 AI Agent: Hyperparameter Optimization Advisor")
print("="*60)

hp_system = """You are an expert in deep learning hyperparameter tuning for medical AI.
Given the model architecture and dataset characteristics, recommend optimal hyperparameters.
Provide SPECIFIC numerical values with scientific justification.
Format as a structured recommendation table."""

hp_prompt = f"""Model: HyCF-TFM (Multi-modal fusion for heart failure prediction)
Architecture:
- Text Branch: ClinicalBERT (768→512), frozen BERT + trainable FC
- ECG Branch: 1D-ResNet (replacing BiLSTM), output 512-dim
- Image Branch: ResNet-18 (pretrained), output 512-dim  
- Fusion: Gated Cross-Modal Attention (embed=512, heads=8)
- Classifier: 1536→256→1 with Dropout

Dataset: {len(train_dataset)} train, {len(test_dataset)} test
Class balance: {n_pos_train} positive ({n_pos_train/len(train_dataset)*100:.1f}%), WeightedSampler active
Previous issues: Overfitting at epoch 3, ECG branch attention near zero, val_loss divergence

Please recommend:
1. Learning rate & scheduler (separate for BERT vs other modules)
2. Focal Loss hyperparameters (gamma, alpha)
3. Dropout rate for classifier
4. Number of epochs with early stopping patience
5. ECG auxiliary loss weight
6. Label smoothing factor
7. Gradient clipping threshold"""

hp_recommendation = agent_call(hp_system, hp_prompt)
print(hp_recommendation)

# ---- Apply recommended hyperparameters ----
print("\n" + "="*60)
print("Applied Hyperparameters (Agent-Informed)")
print("="*60)

HP = {
    'lr_bert': 2e-5,           # Lower LR for pretrained BERT
    'lr_other': 1e-3,          # Higher LR for randomly initialized modules
    'lr_ecg_multiplier': 3.0,  # NEW: ECG branch LR multiplier (asymmetric, 3x other)
    'focal_gamma': 2.0,        # Focal Loss focusing parameter
    'focal_alpha': 0.75,       # Weight for positive class
    'dropout': 0.4,            # Increased dropout (was 0.2)
    'epochs': 12,              # More epochs for staged training
    'patience': 4,             # Early stopping patience (increased for staged training)
    'ecg_aux_weight': 0.3,     # ECG auxiliary loss weight
    'label_smoothing': 0.05,   # Light label smoothing
    'grad_clip': 1.0,          # Gradient clipping
    'weight_decay': 1e-4,      # L2 regularization
    'warmup_epochs': 2,        # LR warmup
    # === NEW: Contrastive Alignment Loss ===
    'contrastive_weight': 0.15,    # InfoNCE contrastive loss weight
    'contrastive_temperature': 0.07,  # Temperature for InfoNCE (CLIP default)
    # === NEW: Gating Regularization ===
    'gate_entropy_weight': 0.05,   # Gating entropy regularization weight (encourages balanced gates)
    # === NEW: Text Masking (break text hegemony) ===
    'text_mask_prob': 0.3,         # 30% random text feature dropout during training
    # === NEW: Staged Training ===
    'staged_epochs': 5,            # First N epochs: ECG+Image only (text branch frozen)
    # === NEW: Agent Closed-Loop ===
    'agent_intervene_every': 3,    # Agent intervenes every N epochs
}

for k, v in HP.items():
    print(f"  {k:25s} = {v}")

In [ ]:
# ====================================
# Cell 4.5: AI Agent — Clinical-Prior-Driven Hyperparameter Tuning
# ====================================
# Methodology Contribution: Prove that clinical-prior-guided HP selection
# converges faster than pure mathematical search (random/grid/Bayesian).
#
# Key insight: "心衰患者ECG变化先于影像学表现" (ECG changes precede imaging changes)
# → This clinical knowledge should inform LR ratios, loss weights, staged training design.

print("="*60)
print("🤖 AI Agent: Clinical-Prior-Driven Hyperparameter Tuning")
print("="*60)

clinical_hp_system = """You are a medical AI researcher who uses CLINICAL DOMAIN KNOWLEDGE
to guide hyperparameter selection. This is fundamentally different from pure mathematical
optimization (grid search, Bayesian optimization) because you leverage:

1. **Pathophysiological timing**: ECG changes precede imaging changes in acute HF
   → ECG branch should have HIGHER learning rate to learn faster
   
2. **Information hierarchy**: Clinical notes are most information-dense but create
   text hegemony → Need text masking and staged training
   
3. **Class imbalance in cardiology**: 30-day readmission is rare (~8.7%) → Need
   aggressive Focal Loss gamma and contrastive alignment
   
4. **Clinical feature interaction**: ECG + notes together are more predictive than
   either alone → Contrastive alignment weight should be substantial

Your task: Provide a SCIENTIFICALLY JUSTIFIED hyperparameter configuration.
For each parameter, cite the clinical reasoning.

Return a JSON object with:
{
  "lr_ecg_multiplier": <float, justified by clinical timing>,
  "contrastive_weight": <float, justified by feature interaction>,
  "gate_entropy_weight": <float, justified by modality balance>,
  "text_mask_prob": <float, justified by information hierarchy>,
  "staged_epochs": <int, justified by learning dynamics>,
  "focal_gamma": <float, justified by class imbalance>,
  "reasoning": {
    "lr_ecg_multiplier": "<clinical justification>",
    "contrastive_weight": "<clinical justification>",
    "gate_entropy_weight": "<clinical justification>",
    "text_mask_prob": "<clinical justification>",
    "staged_epochs": "<clinical justification>",
    "focal_gamma": "<clinical justification>"
  }
}"""

clinical_hp_prompt = f"""Dataset characteristics:
- {n_pos_train} positive (readmitted) / {n_neg_train} negative ({pos_rate:.1f}% positive rate)
- ECG signals: 1-lead, 2000 time steps
- Clinical notes: average ~200 tokens, contain medications, vitals, diagnoses
- Chest X-rays: 224x224, showing pulmonary status

Problem from Experiment 3:
- Text gate value was >0.8, ECG gate <0.05 → Text hegemony
- Model predicted all-negative → Class imbalance not addressed
- Validation loss diverged at epoch 3 → Overfitting

Your clinical knowledge should guide the hyperparameters to fix these issues."""

clinical_hp_response = agent_call(clinical_hp_system, clinical_hp_prompt, max_tokens=1200)
print("Agent Clinical-Prior HP Recommendation:")
print(clinical_hp_response)

# Parse and apply clinical-prior-guided HPs
try:
    json_str = clinical_hp_response.strip()
    start_idx = json_str.find('{')
    # Find matching closing brace (handle nested JSON)
    depth = 0
    end_idx = start_idx
    for j in range(start_idx, len(json_str)):
        if json_str[j] == '{': depth += 1
        elif json_str[j] == '}': depth -= 1
        if depth == 0:
            end_idx = j + 1
            break
    
    clinical_params = json.loads(json_str[start_idx:end_idx])
    
    # Apply with safety clamping
    if 'lr_ecg_multiplier' in clinical_params:
        HP['lr_ecg_multiplier'] = max(1.5, min(5.0, float(clinical_params['lr_ecg_multiplier'])))
    if 'contrastive_weight' in clinical_params:
        HP['contrastive_weight'] = max(0.05, min(0.5, float(clinical_params['contrastive_weight'])))
    if 'gate_entropy_weight' in clinical_params:
        HP['gate_entropy_weight'] = max(0.01, min(0.2, float(clinical_params['gate_entropy_weight'])))
    if 'text_mask_prob' in clinical_params:
        HP['text_mask_prob'] = max(0.1, min(0.5, float(clinical_params['text_mask_prob'])))
    if 'staged_epochs' in clinical_params:
        HP['staged_epochs'] = max(2, min(8, int(clinical_params['staged_epochs'])))
    if 'focal_gamma' in clinical_params:
        HP['focal_gamma'] = max(1.0, min(5.0, float(clinical_params['focal_gamma'])))
    
    print("\n" + "="*60)
    print("✅ Clinical-Prior-Guided HPs Applied:")
    print("="*60)
    for k in ['lr_ecg_multiplier', 'contrastive_weight', 'gate_entropy_weight', 
              'text_mask_prob', 'staged_epochs', 'focal_gamma']:
        print(f"  {k:25s} = {HP[k]}")
    
    # Store reasoning for paper
    if 'reasoning' in clinical_params:
        print("\n📋 Clinical Justification (for paper Methods section):")
        for param, reason in clinical_params['reasoning'].items():
            print(f"  • {param}: {reason}")

except Exception as e:
    print(f"\n⚠ Clinical HP parsing failed ({e}), keeping default HPs")
    print("  Using pre-configured clinical-prior-informed defaults")

print("\n" + "="*60)
print("Final HP Configuration (Clinical-Prior-Guided):")
print("="*60)
for k, v in HP.items():
    print(f"  {k:25s} = {v}")

In [ ]:
# ====================================
# Cell 5: Improved Model Architecture
# ====================================

# ---- Focal Loss ----
class FocalLoss(nn.Module):
    """Focal Loss for imbalanced classification (Lin et al., 2017).
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma=2.0, alpha=0.75, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
    
    def forward(self, logits, targets):
        # Apply label smoothing
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = (1 - p_t) ** self.gamma
        
        # Class-specific alpha weighting
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        
        loss = alpha_t * focal_weight * bce
        return loss.mean()


# ---- 1. TextEngine (same as Exp 3) ----
class TextEngine(nn.Module):
    def __init__(self, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained(model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        except:
            print("Warning: ClinicalBERT unavailable, using DistilBERT.")
            self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
            self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
        self.fc = nn.Linear(768, 512)
    
    def get_tokenizer(self):
        return self.tokenizer
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.fc(cls_token)


# ---- 2. 1D-ResNet for ECG (replaces BiLSTM) ----
class ResBlock1D(nn.Module):
    """1D Residual Block for ECG signal processing."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )
    
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)


class ECGResNet1D(nn.Module):
    """1D-ResNet for ECG time-series. Replaces BiLSTM with CNN-based architecture
    for better temporal pattern extraction.
    Architecture: Conv1D stem → 4 ResBlocks → GlobalAvgPool → FC(512)
    """
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )
        self.layer1 = ResBlock1D(32, 64, stride=2)
        self.layer2 = ResBlock1D(64, 128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 512, stride=2)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, 512)
        
        # Auxiliary classification head (forces ECG branch to learn discriminative features)
        self.aux_classifier = nn.Sequential(
            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    
    def forward(self, x, return_aux=False):
        # x: [B, 1, 2000]
        out = self.stem(x)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.global_pool(out).squeeze(-1)  # [B, 512]
        feat = self.fc(out)
        
        if return_aux:
            aux_logits = self.aux_classifier(out)
            return feat, aux_logits
        return feat


# ---- 3. VisualEngine (same robust import as Exp 3) ----
class VisualEngine(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = None
        try:
            import importlib
            tv_models = importlib.import_module('torchvision.models')
            if hasattr(tv_models, 'ResNet18_Weights'):
                try:
                    weights = getattr(tv_models, 'ResNet18_Weights').IMAGENET1K_V1
                    resnet = tv_models.resnet18(weights=weights)
                except Exception:
                    resnet = tv_models.resnet18(weights=None)
            else:
                try:
                    resnet = tv_models.resnet18(pretrained=False)
                except Exception:
                    resnet = None
        except Exception:
            resnet = None

        if resnet is None:
            self.features = nn.Sequential(
                nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool2d((1, 1)),
                nn.Flatten()
            )
            self.fc = nn.Linear(64, 512)
            print("VisualEngine: Using fallback small CNN.")
        else:
            self.features = nn.Sequential(*list(resnet.children())[:-1])
            self.fc = nn.Linear(512, 512)

    def forward(self, x):
        f = self.features(x)
        if f.dim() > 2:
            f = f.view(f.size(0), -1)
        return self.fc(f)


# ---- 4. Gated Cross-Modal Fusion (Softmax + Entropy Regularization) ----
class GatedCrossModalFusion(nn.Module):
    """Gated Cross-Modal Attention Fusion with SOFTMAX gates (replacing Sigmoid).
    
    Key improvements over v1:
    - Softmax gates enforce competition between modalities (sum=1 constraint)
    - Gating entropy regularization encourages balanced modality contribution
    - Prevents single modality domination (breaks text hegemony)
    """
    def __init__(self, embed_dim=512, num_heads=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.num_layers = num_layers
        
        # Multi-layer cross-modal attention
        self.attn_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(embed_dim) for _ in range(num_layers * 2)])
        self.ffns = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, embed_dim * 2),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim * 2, embed_dim),
                nn.Dropout(dropout)
            ) for _ in range(num_layers)
        ])
        
        # Modality-specific gates: SOFTMAX (sum=1) instead of Sigmoid (independent)
        # This forces explicit modality competition
        self.gate_logits = nn.Sequential(
            nn.Linear(embed_dim * 3, 128),
            nn.GELU(),
            nn.Linear(128, 3)  # 3 modalities
        )
    
    def compute_gate_entropy(self, gate_probs):
        """Compute entropy of gate distribution.
        Higher entropy = more balanced gates = better.
        Used as regularization loss (maximize entropy = minimize negative entropy).
        H(g) = -Σ g_i * log(g_i)
        """
        eps = 1e-8
        entropy = -torch.sum(gate_probs * torch.log(gate_probs + eps), dim=-1)
        return entropy.mean()  # Average over batch
    
    def forward(self, x):
        # x: [B, 3, 512]  (3 modality tokens)
        weights_all = None
        
        for i in range(self.num_layers):
            # Self-attention across modalities
            attn_out, weights = self.attn_layers[i](x, x, x, need_weights=True)
            x = self.norms[i*2](x + attn_out)
            x = self.norms[i*2+1](x + self.ffns[i](x))
            if i == self.num_layers - 1:
                weights_all = weights  # Keep last layer's attention for visualization
        
        # SOFTMAX gated fusion: forces competitive modality selection
        concat_feats = x.reshape(x.size(0), -1)  # [B, 1536]
        gate_raw = self.gate_logits(concat_feats)  # [B, 3]
        gates = F.softmax(gate_raw, dim=-1)        # SOFTMAX: sum=1 (was Sigmoid)
        
        # Apply gates
        gates_expanded = gates.unsqueeze(-1)       # [B, 3, 1]
        x = x * gates_expanded                     # Gated modality features
        
        # Compute gate entropy for regularization
        gate_entropy = self.compute_gate_entropy(gates)
        
        return x, weights_all, gates, gate_entropy


# ---- NEW: Contrastive Alignment Loss (CLIP-style InfoNCE) ----
class ContrastiveAlignmentLoss(nn.Module):
    """CLIP-style InfoNCE contrastive loss for cross-modal alignment.
    
    Forces representations from different modalities of the SAME patient
    to be closer in embedding space than representations from DIFFERENT patients.
    
    This prevents "pseudo-alignment" where modality features drift into
    unrelated subspaces while still achieving low classification loss.
    
    Reference: Radford et al., "Learning Transferable Visual Models From
               Natural Language Supervision", ICML 2021
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        # Learnable projection heads (project to shared contrastive space)
        self.proj_text = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.proj_ecg = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        self.proj_img = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
    
    def forward(self, f_text, f_ecg, f_img):
        """Compute pairwise InfoNCE losses between all modality pairs.
        
        Returns mean of 3 pairwise contrastive losses:
          L_total = (L_text_ecg + L_text_img + L_ecg_img) / 3
        """
        # Project to contrastive space & L2 normalize
        z_text = F.normalize(self.proj_text(f_text), dim=-1)  # [B, 128]
        z_ecg  = F.normalize(self.proj_ecg(f_ecg), dim=-1)
        z_img  = F.normalize(self.proj_img(f_img), dim=-1)
        
        loss_te = self._infonce(z_text, z_ecg)   # text ↔ ecg
        loss_ti = self._infonce(z_text, z_img)    # text ↔ image
        loss_ei = self._infonce(z_ecg, z_img)     # ecg ↔ image
        
        return (loss_te + loss_ti + loss_ei) / 3.0
    
    def _infonce(self, z_a, z_b):
        """Symmetric InfoNCE loss between two views.
        Diagonal entries are positives (same patient), off-diagonal are negatives.
        """
        B = z_a.size(0)
        if B <= 1:
            return torch.tensor(0.0, device=z_a.device)
        
        # Cosine similarity matrix [B, B]
        sim_matrix = torch.matmul(z_a, z_b.T) / self.temperature
        
        # Labels: diagonal entries are positives
        labels = torch.arange(B, device=z_a.device)
        
        # Symmetric loss
        loss_a2b = F.cross_entropy(sim_matrix, labels)
        loss_b2a = F.cross_entropy(sim_matrix.T, labels)
        
        return (loss_a2b + loss_b2a) / 2.0


# ---- 5. HyCF-TFM v2 (Improved with Contrastive Alignment + Text Masking + Intent Attention) ----
class HyCF_TFM_v2(nn.Module):
    """HyCF-TFM v2: Intent-Driven Multimodal Fusion with:
    - 1D-ResNet ECG branch (replacing BiLSTM)
    - Gated Cross-Modal Fusion (Softmax gates + entropy regularization)
    - Contrastive Alignment Loss (CLIP-style InfoNCE)
    - Text Feature Masking (30% dropout to break text hegemony)
    - Intent Attention Vector (Agent-generated clinical prior as attention init)
    - ECG auxiliary classification head
    """
    def __init__(self, dropout=0.4, contrastive_temp=0.07, text_mask_prob=0.3):
        super().__init__()
        self.text_engine = TextEngine()
        self.ecg_engine = ECGResNet1D()        # 1D-ResNet replaces BiLSTM
        self.visual_engine = VisualEngine()
        self.fusion = GatedCrossModalFusion(embed_dim=512, num_heads=8, num_layers=2, dropout=0.1)
        
        # NEW: Contrastive alignment loss module
        self.contrastive_loss_fn = ContrastiveAlignmentLoss(temperature=contrastive_temp)
        
        # NEW: Text mask probability (break text hegemony)
        self.text_mask_prob = text_mask_prob
        
        # NEW: Intent Attention — learnable bias initialized by Agent's clinical prior
        # This vector modulates attention to each modality based on clinical knowledge
        # e.g., "ECG changes precede imaging changes in acute HF"
        self.intent_attention_bias = nn.Parameter(torch.zeros(3, 512))
        nn.init.normal_(self.intent_attention_bias, mean=0, std=0.02)
        
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1)
        )
    
    def get_tokenizer(self):
        return self.text_engine.get_tokenizer()
    
    def set_intent_attention(self, intent_vector):
        """Set the intent attention bias from Agent's clinical prior knowledge.
        intent_vector: [3, 512] tensor or [3] scaling factors for each modality.
        """
        with torch.no_grad():
            if intent_vector.dim() == 1 and intent_vector.size(0) == 3:
                # Scale factor mode: [3] → [3, 512]
                self.intent_attention_bias.copy_(
                    intent_vector.unsqueeze(-1).expand(3, 512) * 0.1
                )
            else:
                self.intent_attention_bias.copy_(intent_vector)
    
    def forward(self, ecg, img, input_ids, mask, return_aux=False, training_phase='full'):
        f_text = self.text_engine(input_ids, mask)
        
        if return_aux:
            f_ecg, ecg_aux_logits = self.ecg_engine(ecg, return_aux=True)
        else:
            f_ecg = self.ecg_engine(ecg)
            ecg_aux_logits = None
        
        f_img = self.visual_engine(img)
        
        # === NEW: Text Feature Masking (training only) ===
        # Randomly mask 30% of text features to force model to learn from ECG/Image
        if self.training and self.text_mask_prob > 0:
            text_mask = torch.bernoulli(
                torch.full_like(f_text, 1.0 - self.text_mask_prob)
            )
            f_text = f_text * text_mask / (1.0 - self.text_mask_prob)  # Scale to maintain magnitude
        
        # === NEW: Staged Training — freeze text during early epochs ===
        if training_phase == 'ecg_img_only':
            f_text = f_text.detach()  # Stop gradient for text (don't update text branch)
        
        # === NEW: Add Intent Attention Bias (Agent clinical prior) ===
        intent_bias = self.intent_attention_bias.unsqueeze(0).expand(f_text.size(0), -1, -1)
        stacked = torch.stack([f_text, f_ecg, f_img], dim=1)  # [B, 3, 512]
        stacked = stacked + intent_bias.to(stacked.device)     # Add clinical intent bias
        
        # Fusion with Softmax gates + entropy regularization
        fused, weights, gates, gate_entropy = self.fusion(stacked)
        
        flat_fused = fused.reshape(fused.size(0), -1)  # [B, 1536]
        logits = self.classifier(flat_fused)
        
        # === NEW: Compute Contrastive Alignment Loss ===
        contrastive_loss = self.contrastive_loss_fn(f_text, f_ecg, f_img)
        
        if return_aux:
            return logits, weights, gates, ecg_aux_logits, contrastive_loss, gate_entropy
        return logits, weights, gates, contrastive_loss, gate_entropy


# ---- 6. Concat Baseline v2 (same improvements for fair comparison) ----
class ConcatFusionV2(nn.Module):
    """Improved concat baseline with same ECG branch for fair comparison."""
    def __init__(self, dropout=0.4):
        super().__init__()
        self.text_engine = TextEngine()
        self.ecg_engine = ECGResNet1D()
        self.visual_engine = VisualEngine()
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1)
        )
    
    def get_tokenizer(self):
        return self.text_engine.get_tokenizer()
    
    def forward(self, ecg, img, input_ids, mask, return_aux=False, training_phase='full'):
        f_text = self.text_engine(input_ids, mask)
        f_ecg = self.ecg_engine(ecg)
        f_img = self.visual_engine(img)
        concat = torch.cat([f_text, f_ecg, f_img], dim=1)
        dummy = torch.tensor(0.0, device=ecg.device)
        return self.classifier(concat), None, None, dummy, dummy


print("Improved model architectures defined:")
print("  - FocalLoss (gamma=2, alpha=0.75)")
print("  - ECGResNet1D (1D-ResNet with auxiliary head)")
print("  - ContrastiveAlignmentLoss (CLIP-style InfoNCE)")
print("  - GatedCrossModalFusion (Softmax gates + entropy regularization)")
print("  - HyCF_TFM_v2 (contrastive alignment + text masking + intent attention)")
print("  - ConcatFusionV2 (fair comparison baseline)")

In [ ]:
# ====================================
# Cell 6: Training Engine with AI Agent Monitoring
# ====================================

def train_v2(model, train_loader, test_loader, device, hp, model_name="Model",
             use_aux_loss=False, agent_monitor=True):
    """
    Enhanced training loop v2 with:
    - Focal Loss + Contrastive Alignment Loss (InfoNCE)
    - Softmax gates + gating entropy regularization
    - Differential learning rates (BERT vs ECG vs others)
    - Asymmetric ECG LR (3x other) to boost ECG branch
    - Staged training: first N epochs ECG+Image only, then full
    - 30% text feature masking during training
    - Agent closed-loop intervention every 3 epochs
    - ECG auxiliary loss
    - Cosine annealing LR scheduler + early stopping
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    model = model.to(device)
    tokenizer = model.get_tokenizer()
    
    # ---- Asymmetric learning rates: BERT < Other < ECG ----
    bert_params = []
    ecg_params = []
    contrastive_params = []
    other_params = []
    for name, param in model.named_parameters():
        if 'bert' in name.lower():
            bert_params.append(param)
        elif 'ecg_engine' in name.lower():
            ecg_params.append(param)
        elif 'contrastive_loss_fn' in name.lower():
            contrastive_params.append(param)
        else:
            other_params.append(param)
    
    ecg_lr = hp['lr_other'] * hp.get('lr_ecg_multiplier', 3.0)
    
    optimizer = optim.AdamW([
        {'params': bert_params, 'lr': hp['lr_bert']},
        {'params': ecg_params, 'lr': ecg_lr},              # ECG gets higher LR
        {'params': contrastive_params, 'lr': hp['lr_other']},
        {'params': other_params, 'lr': hp['lr_other']}
    ], weight_decay=hp['weight_decay'])
    
    print(f"  Learning Rates: BERT={hp['lr_bert']}, ECG={ecg_lr}, Other={hp['lr_other']}")
    print(f"  Staged Training: First {hp.get('staged_epochs', 5)} epochs ECG+Image only")
    print(f"  Text Mask Prob: {hp.get('text_mask_prob', 0.3)}")
    print(f"  Agent Intervention: Every {hp.get('agent_intervene_every', 3)} epochs")
    
    # ---- Cosine Annealing LR with Warmup ----
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=hp['epochs'], T_mult=1, eta_min=1e-6
    )
    
    # ---- Loss functions ----
    criterion = FocalLoss(
        gamma=hp['focal_gamma'],
        alpha=hp['focal_alpha'],
        label_smoothing=hp['label_smoothing']
    )
    aux_criterion = nn.BCEWithLogitsLoss()  # For ECG auxiliary task
    
    # ---- Training history ----
    history = {
        'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_ap': [],
        'val_f1': [], 'val_precision': [], 'val_recall': [],
        'lr': [], 'ecg_aux_loss': [], 'gate_values': [],
        'contrastive_loss': [], 'gate_entropy': [],       # NEW tracking
        'training_phase': [], 'agent_interventions': []   # NEW tracking
    }
    
    best_val_auc = 0.0
    best_model_state = None
    patience_counter = 0
    staged_epochs = hp.get('staged_epochs', 5)
    agent_intervene_every = hp.get('agent_intervene_every', 3)
    
    for epoch in range(hp['epochs']):
        # ========== DETERMINE TRAINING PHASE ==========
        if epoch < staged_epochs:
            training_phase = 'ecg_img_only'
            phase_label = f'Stage 1 (ECG+Image only, epoch {epoch+1}/{staged_epochs})'
        else:
            training_phase = 'full'
            phase_label = f'Stage 2 (Full multimodal, epoch {epoch+1})'
        
        history['training_phase'].append(training_phase)
        print(f"\n  [{phase_label}]")
        
        # ========== AGENT CLOSED-LOOP INTERVENTION ==========
        if agent_monitor and epoch > 0 and epoch % agent_intervene_every == 0:
            print(f"  🤖 Agent Closed-Loop Intervention (Epoch {epoch+1})...")
            
            intervention_system = """You are a deep learning training optimizer AI.
Analyze the training trajectory and return a JSON object with EXACTLY these keys:
{
  "lr_scale": <float 0.1-2.0, multiplier for current learning rate>,
  "focal_gamma": <float 1.0-5.0, Focal Loss gamma>,
  "contrastive_weight": <float 0.05-0.5, weight for contrastive loss>,
  "gate_entropy_weight": <float 0.01-0.2, weight for gate entropy regularization>,
  "text_mask_prob": <float 0.1-0.5, text feature masking probability>,
  "reasoning": "<one sentence explaining your adjustments>"
}

Clinical prior knowledge to consider:
- In acute heart failure, ECG changes (ST depression, T-wave inversion, arrhythmia) 
  typically PRECEDE imaging changes (pulmonary edema on CXR)
- If ECG gate values are low, increase contrastive_weight and text_mask_prob
- If val_loss is increasing while train_loss decreases → overfitting → reduce lr_scale
- If val_AUC plateaus → try increasing focal_gamma to focus on hard samples
- If gate entropy is low → one modality dominates → increase gate_entropy_weight

Return ONLY the JSON object, no other text."""
            
            recent_history = {
                'train_loss': history['train_loss'][-min(3, len(history['train_loss'])):],
                'val_loss': history['val_loss'][-min(3, len(history['val_loss'])):],
                'val_auc': history['val_auc'][-min(3, len(history['val_auc'])):],
                'val_f1': history['val_f1'][-min(3, len(history['val_f1'])):],
                'gate_values': history['gate_values'][-min(3, len(history['gate_values'])):],
                'contrastive_loss': history['contrastive_loss'][-min(3, len(history['contrastive_loss'])):],
                'gate_entropy': history['gate_entropy'][-min(3, len(history['gate_entropy'])):],
                'current_epoch': epoch + 1,
                'total_epochs': hp['epochs'],
                'training_phase': training_phase,
                'current_lr_other': optimizer.param_groups[3]['lr'],
                'current_lr_ecg': optimizer.param_groups[1]['lr'],
            }
            
            intervention_prompt = f"Training state:\n{json.dumps(recent_history, indent=2)}"
            
            agent_response = agent_call(intervention_system, intervention_prompt, max_tokens=500)
            
            # Parse Agent's JSON response
            try:
                # Extract JSON from response
                json_str = agent_response.strip()
                if '```' in json_str:
                    json_str = json_str.split('```')[1]
                    if json_str.startswith('json'):
                        json_str = json_str[4:]
                # Find JSON object
                start_idx = json_str.find('{')
                end_idx = json_str.rfind('}') + 1
                if start_idx >= 0 and end_idx > start_idx:
                    json_str = json_str[start_idx:end_idx]
                
                params = json.loads(json_str)
                
                # Apply Agent's adjustments
                lr_scale = float(params.get('lr_scale', 1.0))
                lr_scale = max(0.1, min(2.0, lr_scale))  # Clamp
                
                for pg in optimizer.param_groups:
                    pg['lr'] = pg['lr'] * lr_scale
                
                new_gamma = float(params.get('focal_gamma', hp['focal_gamma']))
                criterion.gamma = max(1.0, min(5.0, new_gamma))
                
                new_contrastive_w = float(params.get('contrastive_weight', hp['contrastive_weight']))
                hp['contrastive_weight'] = max(0.05, min(0.5, new_contrastive_w))
                
                new_gate_entropy_w = float(params.get('gate_entropy_weight', hp['gate_entropy_weight']))
                hp['gate_entropy_weight'] = max(0.01, min(0.2, new_gate_entropy_w))
                
                new_text_mask = float(params.get('text_mask_prob', hp['text_mask_prob']))
                hp['text_mask_prob'] = max(0.1, min(0.5, new_text_mask))
                if hasattr(model, 'text_mask_prob'):
                    model.text_mask_prob = hp['text_mask_prob']
                
                reasoning = params.get('reasoning', 'No reasoning provided')
                
                print(f"    ✅ Agent Adjustments Applied:")
                print(f"       LR scale: {lr_scale:.2f}, Focal γ: {criterion.gamma:.1f}")
                print(f"       Contrastive weight: {hp['contrastive_weight']:.3f}")
                print(f"       Gate entropy weight: {hp['gate_entropy_weight']:.3f}")
                print(f"       Text mask prob: {hp['text_mask_prob']:.2f}")
                print(f"       Reasoning: {reasoning}")
                
                history['agent_interventions'].append({
                    'epoch': epoch + 1,
                    'params': params,
                    'reasoning': reasoning
                })
                
            except Exception as e:
                print(f"    ⚠ Agent response parse failed: {e}")
                print(f"    Raw response: {agent_response[:200]}")
                history['agent_interventions'].append({
                    'epoch': epoch + 1,
                    'error': str(e)
                })
        
        # ========== TRAIN ==========
        model.train()
        total_loss, total_aux_loss = 0, 0
        total_contrastive_loss, total_gate_entropy = 0, 0
        train_preds, train_targets = [], []
        epoch_gates = []
        
        for batch in train_loader:
            ecg = batch['ecg'].to(device)
            img = batch['image'].to(device)
            targets = batch['label'].to(device).unsqueeze(1)
            texts = batch['text_raw']
            inputs = tokenizer(list(texts), padding=True, truncation=True,
                             return_tensors='pt', max_length=128)
            input_ids = inputs['input_ids'].to(device)
            mask = inputs['attention_mask'].to(device)
            
            optimizer.zero_grad()
            
            if use_aux_loss:
                logits, weights, gates, ecg_aux, contrastive_loss, gate_entropy = model(
                    ecg, img, input_ids, mask, return_aux=True, training_phase=training_phase
                )
                # Main classification loss
                main_loss = criterion(logits, targets)
                # ECG auxiliary loss (forces ECG branch to learn)
                aux_loss = aux_criterion(ecg_aux, targets) if ecg_aux is not None else torch.tensor(0.0)
                # Total loss = classification + ECG_aux + contrastive + gate_entropy_reg
                loss = (main_loss 
                        + hp['ecg_aux_weight'] * aux_loss
                        + hp['contrastive_weight'] * contrastive_loss
                        - hp['gate_entropy_weight'] * gate_entropy)  # NEGATIVE: maximize entropy
                
                total_aux_loss += (aux_loss.item() if isinstance(aux_loss, torch.Tensor) else 0)
                total_contrastive_loss += contrastive_loss.item()
                total_gate_entropy += gate_entropy.item()
            else:
                output = model(ecg, img, input_ids, mask, training_phase=training_phase)
                logits = output[0]
                gates = output[2] if len(output) > 2 and output[2] is not None else None
                contrastive_loss = output[3] if len(output) > 3 else torch.tensor(0.0)
                gate_entropy = output[4] if len(output) > 4 else torch.tensor(0.0)
                
                main_loss = criterion(logits, targets)
                loss = (main_loss
                        + hp['contrastive_weight'] * contrastive_loss
                        - hp['gate_entropy_weight'] * gate_entropy)
                
                total_contrastive_loss += contrastive_loss.item()
                total_gate_entropy += gate_entropy.item()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp['grad_clip'])
            optimizer.step()
            
            total_loss += loss.item()
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            train_preds.extend(probs.flatten())
            train_targets.extend(targets.cpu().numpy().flatten())
            
            if gates is not None:
                epoch_gates.append(gates.detach().cpu().numpy())
        
        scheduler.step()
        
        n_batches = max(len(train_loader), 1)
        avg_train_loss = total_loss / n_batches
        avg_contrastive = total_contrastive_loss / n_batches
        avg_gate_entropy = total_gate_entropy / n_batches
        
        try:
            train_auc = roc_auc_score(train_targets, train_preds) if len(np.unique(train_targets)) > 1 else 0.5
        except:
            train_auc = 0.5
        
        # ========== EVALUATE ==========
        model.eval()
        val_loss_total = 0
        val_preds, val_targets = [], []
        
        with torch.no_grad():
            for batch in test_loader:
                ecg = batch['ecg'].to(device)
                img = batch['image'].to(device)
                targets = batch['label'].to(device).unsqueeze(1)
                texts = batch['text_raw']
                inputs = tokenizer(list(texts), padding=True, truncation=True,
                                 return_tensors='pt', max_length=128)
                input_ids = inputs['input_ids'].to(device)
                mask = inputs['attention_mask'].to(device)
                
                output = model(ecg, img, input_ids, mask, training_phase='full')
                logits = output[0]
                loss = criterion(logits, targets)
                val_loss_total += loss.item()
                
                probs = torch.sigmoid(logits).cpu().numpy()
                val_preds.extend(probs.flatten())
                val_targets.extend(targets.cpu().numpy().flatten())
        
        avg_val_loss = val_loss_total / max(len(test_loader), 1)
        
        # Compute metrics
        try:
            if len(np.unique(val_targets)) > 1:
                val_auc = roc_auc_score(val_targets, val_preds)
                val_ap = average_precision_score(val_targets, val_preds)
                pred_binary = (np.array(val_preds) > 0.5).astype(int)
                val_f1 = f1_score(val_targets, pred_binary)
                val_prec = precision_score(val_targets, pred_binary, zero_division=0)
                val_rec = recall_score(val_targets, pred_binary, zero_division=0)
            else:
                val_auc, val_ap, val_f1, val_prec, val_rec = 0.5, 0.0, 0.0, 0.0, 0.0
        except:
            val_auc, val_ap, val_f1, val_prec, val_rec = 0.5, 0.0, 0.0, 0.0, 0.0
        
        # Store history
        current_lr = optimizer.param_groups[3]['lr']  # 'other' group LR
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
        history['val_ap'].append(val_ap)
        history['val_f1'].append(val_f1)
        history['val_precision'].append(val_prec)
        history['val_recall'].append(val_rec)
        history['lr'].append(current_lr)
        history['ecg_aux_loss'].append(total_aux_loss / n_batches)
        history['contrastive_loss'].append(avg_contrastive)
        history['gate_entropy'].append(avg_gate_entropy)
        
        if epoch_gates:
            gate_mean = np.concatenate(epoch_gates, axis=0).mean(axis=0)
            history['gate_values'].append(gate_mean.tolist())
        else:
            history['gate_values'].append([0.33, 0.33, 0.33])
        
        # Print progress
        gate_str = ""
        if history['gate_values'][-1] != [0.33, 0.33, 0.33]:
            g = history['gate_values'][-1]
            gate_str = f" | Gates: T={g[0]:.2f} E={g[1]:.2f} I={g[2]:.2f}"
        
        print(f"  Epoch {epoch+1}/{hp['epochs']} | "
              f"Loss: {avg_train_loss:.4f} (CL:{avg_contrastive:.3f} GE:{avg_gate_entropy:.3f}) | "
              f"Val AUC: {val_auc:.3f} F1: {val_f1:.3f} P: {val_prec:.3f} R: {val_rec:.3f}{gate_str}")
        
        # ---- Early Stopping ----
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= hp['patience']:
                print(f"  ⚠ Early stopping at epoch {epoch+1} (patience={hp['patience']})")
                break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"  ✓ Restored best model (Val AUC: {best_val_auc:.4f})")
    
    # ========== AI Agent: Training Analysis ==========
    if agent_monitor:
        print(f"\n  🤖 AI Agent Training Analysis for {model_name}:")
        monitor_system = """You are a deep learning training diagnostics expert.
Analyze the training trajectory with special attention to:
1. **Staged Training Effectiveness**: Did the ECG+Image-only stage help ECG branch?
2. **Contrastive Loss Trend**: Is cross-modal alignment improving?
3. **Gate Entropy**: Are modalities contributing more equally over time?
4. **Agent Interventions**: Did the closed-loop adjustments help?
5. **Overall Convergence**: Final assessment and remaining issues.
Be concise (8-12 sentences max)."""
        
        monitor_prompt = f"""Model: {model_name}
Training History (per epoch):
  Train Loss: {history['train_loss']}
  Val Loss:   {history['val_loss']}
  Val AUC:    {history['val_auc']}
  Val F1:     {history['val_f1']}
  Gate Values: {history['gate_values']}
  Contrastive Loss: {history['contrastive_loss']}
  Gate Entropy:     {history['gate_entropy']}
  Training Phases:  {history['training_phase']}
  Agent Interventions: {json.dumps(history['agent_interventions'])}
  Best AUC: {best_val_auc:.4f}
  Staged Epochs: {staged_epochs} (ECG+Image only)"""
        
        analysis = agent_call(monitor_system, monitor_prompt, max_tokens=600)
        print(f"  {analysis}")
    
    return model, tokenizer, history, val_targets, val_preds

print("Enhanced training engine v2 defined.")
print("Features: Focal Loss, Contrastive Alignment Loss (InfoNCE),")
print("          Softmax Gates + Entropy Regularization, Asymmetric ECG LR,")
print("          Staged Training (ECG+Image first), 30% Text Masking,")
print("          Agent Closed-Loop Intervention, Early Stopping")

In [ ]:
# ====================================
# Cell 7: Train HyCF-TFM v2 (Main Model)
# ====================================

model_v2 = HyCF_TFM_v2(
    dropout=HP['dropout'],
    contrastive_temp=HP['contrastive_temperature'],
    text_mask_prob=HP['text_mask_prob']
)

# === NEW: Agent-Generated Intent Attention Vector ===
# Ask AI Agent to generate clinical-prior-driven modality importance weights
print("🤖 AI Agent: Generating Intent Attention Vector from Clinical Prior...")
intent_system = """You are a cardiologist with expertise in heart failure.
Based on clinical evidence, rate the RELATIVE IMPORTANCE of each data modality 
for predicting 30-day hospital readmission in heart failure patients.

Return ONLY a JSON object: {"text": <float>, "ecg": <float>, "image": <float>}
Values should be between 0.0 and 1.0, reflecting clinical importance.

Clinical considerations:
- ECG changes (arrhythmia, ST changes, QT prolongation) often PRECEDE symptomatic deterioration
- Clinical notes contain medication history, comorbidities, and social factors
- Chest X-rays show pulmonary congestion but are less sensitive for early readmission risk
- Patients with ECG abnormalities at discharge have 2-3x higher readmission risk"""

intent_prompt = "Rate modality importance for 30-day heart failure readmission prediction."
intent_response = agent_call(intent_system, intent_prompt, max_tokens=200)

try:
    json_str = intent_response.strip()
    start_idx = json_str.find('{')
    end_idx = json_str.rfind('}') + 1
    if start_idx >= 0 and end_idx > start_idx:
        intent_weights = json.loads(json_str[start_idx:end_idx])
    else:
        intent_weights = {"text": 0.35, "ecg": 0.45, "image": 0.20}
    
    intent_vector = torch.tensor([
        intent_weights.get('text', 0.35),
        intent_weights.get('ecg', 0.45),
        intent_weights.get('image', 0.20)
    ], dtype=torch.float32)
    
    model_v2.set_intent_attention(intent_vector)
    print(f"  Intent Attention Vector set: Text={intent_vector[0]:.2f}, "
          f"ECG={intent_vector[1]:.2f}, Image={intent_vector[2]:.2f}")
    print(f"  Agent rationale: {intent_response[:200]}")
except Exception as e:
    print(f"  ⚠ Intent vector parse failed ({e}), using learned initialization")

# === Train with all improvements ===
model_v2, tok_v2, hist_v2, yt_v2, yp_v2 = train_v2(
    model_v2, train_loader, test_loader, device, HP,
    model_name="HyCF-TFM v2 (Intent-Driven Multimodal)",
    use_aux_loss=True,
    agent_monitor=True
)

# Save improved model
torch.save(model_v2.state_dict(), os.path.join(BASE, "hycf_tfm_v2_model.pth"))
print(f"\nModel saved to {os.path.join(BASE, 'hycf_tfm_v2_model.pth')}")

In [ ]:
# ====================================
# Cell 8: Train Concat Baseline v2 (Fair Comparison)
# ====================================

model_concat_v2 = ConcatFusionV2(dropout=HP['dropout'])

model_concat_v2, tok_concat, hist_concat, yt_concat, yp_concat = train_v2(
    model_concat_v2, train_loader, test_loader, device, HP,
    model_name="Concat-Fusion v2 (No Attention)",
    use_aux_loss=False,
    agent_monitor=True
)

In [ ]:
# ====================================
# Cell 9: Comprehensive Comparison Visualization
# ====================================

results_v2 = {
    'HyCF-TFM v2 (Ours)': {'history': hist_v2, 'y_true': yt_v2, 'y_pred': yp_v2},
    'Concat-Fusion v2': {'history': hist_concat, 'y_true': yt_concat, 'y_pred': yp_concat},
}

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
colors_v2 = ['#E53935', '#1E88E5', '#43A047', '#FB8C00', '#8E24AA']

# --- A. ROC Curves ---
ax = axes[0, 0]
for i, (name, data) in enumerate(results_v2.items()):
    fpr, tpr, _ = roc_curve(data['y_true'], data['y_pred'])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors_v2[i], lw=2.5, label=f'{name} (AUC={roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('A. ROC Curves — Improved Models', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.grid(True, alpha=0.3)

# --- B. Precision-Recall Curve ---
ax = axes[0, 1]
for i, (name, data) in enumerate(results_v2.items()):
    prec_arr, rec_arr, _ = precision_recall_curve(data['y_true'], data['y_pred'])
    ap_val = average_precision_score(data['y_true'], data['y_pred'])
    ax.plot(rec_arr, prec_arr, color=colors_v2[i], lw=2.5, label=f'{name} (AP={ap_val:.3f})')
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('B. Precision-Recall Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)

# --- C. Confusion Matrix (HyCF-TFM v2) ---
ax = axes[0, 2]
y_pred_binary = (np.array(yt_v2) > 0).astype(int)
yp_binary = (np.array(yp_v2) > 0.5).astype(int)
cm = confusion_matrix(y_pred_binary, yp_binary)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Low Risk', 'High Risk'], yticklabels=['Low Risk', 'High Risk'])
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
ax.set_title('C. Confusion Matrix (HyCF-TFM v2)', fontsize=13, fontweight='bold')

# --- D. Training & Val Loss Curves ---
ax = axes[1, 0]
for i, (name, data) in enumerate(results_v2.items()):
    epochs_range = range(1, len(data['history']['train_loss'])+1)
    ax.plot(epochs_range, data['history']['train_loss'], '--', color=colors_v2[i], alpha=0.5, label=f'{name} (train)')
    ax.plot(epochs_range, data['history']['val_loss'], '-', color=colors_v2[i], lw=2, label=f'{name} (val)')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Focal Loss', fontsize=11)
ax.set_title('D. Loss Curves (Dashed=Train, Solid=Val)', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- E. Gate Values + Contrastive Loss + Gate Entropy Over Epochs ---
ax = axes[1, 1]
if hist_v2['gate_values'] and hist_v2['gate_values'][0] != [0.33, 0.33, 0.33]:
    gate_arr = np.array(hist_v2['gate_values'])
    epochs_range = range(1, len(gate_arr)+1)
    ax.plot(epochs_range, gate_arr[:, 0], 'o-', color='#1E88E5', lw=2, label='Text Gate', markersize=6)
    ax.plot(epochs_range, gate_arr[:, 1], 's-', color='#E53935', lw=2, label='ECG Gate', markersize=6)
    ax.plot(epochs_range, gate_arr[:, 2], '^-', color='#43A047', lw=2, label='Image Gate', markersize=6)
    
    # Mark staged training boundary
    staged_epochs = HP.get('staged_epochs', 5)
    if staged_epochs < len(gate_arr):
        ax.axvline(x=staged_epochs + 0.5, color='red', linestyle=':', alpha=0.7, label='Stage 1→2 Boundary')
    
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Gate Value (Softmax)', fontsize=11)
    ax.set_title('E. Modality Gate Evolution\n(Softmax + Staged Training)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Gate values not available', ha='center', va='center', fontsize=14)
    ax.set_title('E. Modality Gate Evolution', fontsize=13, fontweight='bold')

# --- F. Metric Summary Table ---
ax = axes[1, 2]
ax.axis('off')
table_data = []
for name, data in results_v2.items():
    h = data['history']
    table_data.append([
        name,
        f"{max(h['val_auc']):.3f}",
        f"{max(h['val_f1']):.3f}",
        f"{max(h['val_ap']):.3f}",
        f"{max(h['val_precision']):.3f}",
        f"{max(h['val_recall']):.3f}"
    ])

table = ax.table(cellText=table_data,
                 colLabels=['Model', 'AUC', 'F1', 'AP', 'Precision', 'Recall'],
                 loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#E3F2FD')
ax.set_title('F. Performance Summary (Best per Metric)', fontsize=13, fontweight='bold', pad=20)

plt.suptitle('Experiment 4: Intent-Driven HyCF-TFM Training Results\n(Contrastive Alignment + Softmax Gates + Staged Training + Agent Closed-Loop)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_exp4_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

# ---- Print Summary ----
print("\n" + "="*70)
print("Table 3: Experiment 4 — Improved Model Performance")
print("="*70)
print(f"{'Model':30s} {'AUC':>7s} {'F1':>7s} {'AP':>7s} {'Prec':>7s} {'Rec':>7s}")
print("-"*70)
for name, data in results_v2.items():
    h = data['history']
    print(f"{name:30s} {max(h['val_auc']):7.3f} {max(h['val_f1']):7.3f} "
          f"{max(h['val_ap']):7.3f} {max(h['val_precision']):7.3f} {max(h['val_recall']):7.3f}")
print("="*70)

In [ ]:
# ====================================
# Cell 9.5: Contrastive Loss & Gate Entropy Evolution + Agent Intervention Timeline
# ====================================

fig, axes = plt.subplots(1, 3, figsize=(22, 5))

# --- A. Contrastive Alignment Loss ---
ax = axes[0]
if hist_v2.get('contrastive_loss'):
    cl = hist_v2['contrastive_loss']
    epochs_range = range(1, len(cl)+1)
    ax.plot(epochs_range, cl, 'o-', color='#7B1FA2', lw=2.5, markersize=6, label='Contrastive Loss (InfoNCE)')
    ax.fill_between(epochs_range, cl, alpha=0.15, color='#7B1FA2')
    
    # Mark agent interventions
    for intervention in hist_v2.get('agent_interventions', []):
        e = intervention.get('epoch', 0)
        if e <= len(cl):
            ax.axvline(x=e, color='orange', linestyle='--', alpha=0.7)
            ax.annotate('🤖', (e, cl[e-1]), fontsize=14, ha='center')
    
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Contrastive Loss', fontsize=11)
    ax.set_title('A. Cross-Modal Alignment\n(InfoNCE Contrastive Loss ↓ = better alignment)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Contrastive loss not tracked', ha='center', va='center')

# --- B. Gate Entropy Evolution ---
ax = axes[1]
if hist_v2.get('gate_entropy'):
    ge = hist_v2['gate_entropy']
    epochs_range = range(1, len(ge)+1)
    ax.plot(epochs_range, ge, 's-', color='#00897B', lw=2.5, markersize=6, label='Gate Entropy')
    ax.fill_between(epochs_range, ge, alpha=0.15, color='#00897B')
    
    # Maximum possible entropy for 3 modalities = log(3) ≈ 1.099
    max_entropy = np.log(3)
    ax.axhline(y=max_entropy, color='red', linestyle=':', alpha=0.5, label=f'Max Entropy (log3={max_entropy:.3f})')
    
    # Mark staged training boundary
    staged_epochs = HP.get('staged_epochs', 5)
    if staged_epochs < len(ge):
        ax.axvline(x=staged_epochs + 0.5, color='red', linestyle=':', alpha=0.7, label='Stage 1→2')
    
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Gate Entropy (nats)', fontsize=11)
    ax.set_title('B. Gate Balance\n(Higher Entropy = More Balanced Modalities)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Gate entropy not tracked', ha='center', va='center')

# --- C. Agent Intervention Timeline ---
ax = axes[2]
interventions = hist_v2.get('agent_interventions', [])
if interventions:
    epochs_int = [i.get('epoch', 0) for i in interventions]
    params_list = [i.get('params', {}) for i in interventions]
    
    # Plot parameter changes over interventions
    lr_scales = [p.get('lr_scale', 1.0) for p in params_list]
    cl_weights = [p.get('contrastive_weight', HP['contrastive_weight']) for p in params_list]
    ge_weights = [p.get('gate_entropy_weight', HP['gate_entropy_weight']) for p in params_list]
    
    x_pos = range(len(epochs_int))
    width = 0.25
    ax.bar([x - width for x in x_pos], lr_scales, width, label='LR Scale', color='#E53935', alpha=0.7)
    ax.bar(x_pos, cl_weights, width, label='Contrastive W', color='#1E88E5', alpha=0.7)
    ax.bar([x + width for x in x_pos], ge_weights, width, label='Gate Entropy W', color='#43A047', alpha=0.7)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'Epoch {e}' for e in epochs_int], fontsize=9)
    ax.set_ylabel('Parameter Value', fontsize=11)
    ax.set_title('C. Agent Closed-Loop Interventions\n(Dynamic HP Adjustments)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
else:
    ax.text(0.5, 0.5, 'No agent interventions recorded', ha='center', va='center', fontsize=12)
    ax.set_title('C. Agent Interventions', fontsize=12, fontweight='bold')

plt.suptitle('Experiment 4: Alignment & Balance Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_exp4_alignment_diagnostics.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Saved: fig_exp4_alignment_diagnostics.png")

In [ ]:
# ====================================
# Cell 10: Cross-Modal Attention Heatmap (v2)
# ====================================

model_v2.eval()
all_weights = []
all_gates = []

with torch.no_grad():
    for batch in test_loader:
        ecg = batch['ecg'].to(device)
        img = batch['image'].to(device)
        texts = batch['text_raw']
        inputs = tok_v2(list(texts), padding=True, truncation=True, return_tensors='pt', max_length=128)
        ids = inputs['input_ids'].to(device)
        msk = inputs['attention_mask'].to(device)
        
        output = model_v2(ecg, img, ids, msk)
        logits, weights, gates = output[0], output[1], output[2]
        
        # Robust shape handling
        if weights is not None:
            if weights.dim() == 4:
                curr_w = weights.mean(dim=1).cpu().numpy()
            elif weights.dim() == 3:
                curr_w = weights.cpu().numpy()
            else:
                curr_w = weights.reshape(weights.size(0), 3, 3).cpu().numpy()
            all_weights.append(curr_w)
        
        if gates is not None:
            all_gates.append(gates.cpu().numpy())

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
modality_names = ['Text', 'ECG', 'Image']

# --- A. Attention Heatmap ---
ax = axes[0]
if all_weights:
    all_weights_arr = np.concatenate(all_weights, axis=0)
    mean_attn = all_weights_arr.mean(axis=0)
    sns.heatmap(mean_attn, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                xticklabels=modality_names, yticklabels=modality_names, vmin=0, vmax=1)
    ax.set_xlabel('Source Modality (Key)')
    ax.set_ylabel('Target Modality (Query)')
else:
    ax.text(0.5, 0.5, 'Attention weights unavailable', ha='center', va='center')
ax.set_title('A. Cross-Modal Attention Matrix\n(HyCF-TFM v2)', fontsize=12, fontweight='bold')

# --- B. Gate Values Distribution ---
ax = axes[1]
if all_gates:
    all_gates_arr = np.concatenate(all_gates, axis=0)  # [N, 3]
    bp = ax.boxplot([all_gates_arr[:, 0], all_gates_arr[:, 1], all_gates_arr[:, 2]],
                    labels=modality_names, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7))
    gate_colors = ['#1E88E5', '#E53935', '#43A047']
    for patch, color in zip(bp['boxes'], gate_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.5)
    ax.set_ylabel('Gate Value', fontsize=11)
    ax.set_title('B. Modality Gate Distribution\n(Test Set)', fontsize=12, fontweight='bold')
    # Annotate means
    means = all_gates_arr.mean(axis=0)
    for i, m in enumerate(means):
        ax.text(i+1, m + 0.02, f'μ={m:.3f}', ha='center', fontsize=10, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'Gate values unavailable', ha='center', va='center')
    ax.set_title('B. Modality Gate Distribution', fontsize=12, fontweight='bold')

# --- C. Modality Importance Comparison ---
ax = axes[2]
if all_gates:
    importance = all_gates_arr.mean(axis=0)
    importance_norm = importance / (importance.sum() + 1e-9)
    colors_pie = ['#1E88E5', '#E53935', '#43A047']
    wedges, texts_pie, autotexts = ax.pie(
        importance_norm, labels=modality_names, autopct='%1.1f%%',
        colors=colors_pie, startangle=90, textprops={'fontsize': 12}
    )
    for t in autotexts:
        t.set_fontweight('bold')
else:
    ax.text(0.5, 0.5, 'Importance unavailable', ha='center', va='center')
ax.set_title('C. Modality Importance\n(Gate-Weighted)', fontsize=12, fontweight='bold')

plt.suptitle('Cross-Modal Attention & Gating Analysis — HyCF-TFM v2', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_exp4_attention.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ====================================
# Cell 11: AI Agent — Attention Interpretation & Clinical Insight
# ====================================

print("="*60)
print("🤖 AI Agent: Cross-Modal Attention Interpretation")
print("="*60)

# Prepare attention statistics for Agent
attn_stats = {}
if all_weights:
    attn_stats['mean_attention_matrix'] = mean_attn.tolist()
    attn_stats['text_self_attn'] = float(mean_attn[0, 0])
    attn_stats['ecg_self_attn'] = float(mean_attn[1, 1])
    attn_stats['img_self_attn'] = float(mean_attn[2, 2])
    attn_stats['text_to_ecg'] = float(mean_attn[0, 1])
    attn_stats['ecg_to_text'] = float(mean_attn[1, 0])

if all_gates:
    gate_means = all_gates_arr.mean(axis=0)
    attn_stats['gate_text'] = float(gate_means[0])
    attn_stats['gate_ecg'] = float(gate_means[1])
    attn_stats['gate_image'] = float(gate_means[2])

interpret_system = """You are a medical AI interpretability expert. Analyze the cross-modal attention patterns
and gating values of a multimodal heart failure prediction model.

Provide:
1. **Attention Pattern Analysis**: What do the attention weights tell us about inter-modality information flow?
2. **Gate Value Interpretation**: Which modalities does the model find most discriminative?
3. **ECG Branch Assessment**: Has the ECG branch improved compared to Experiment 3 (where it was nearly ignored)?
4. **Clinical Plausibility**: Are these attention patterns clinically reasonable for heart failure prediction?
5. **Improvement Suggestions**: Any further architectural changes recommended?

Be specific and reference the numerical values."""

interpret_prompt = f"""Cross-Modal Attention Analysis Results:
{json.dumps(attn_stats, indent=2)}

Context:
- Model: HyCF-TFM v2 with Gated Cross-Modal Fusion
- ECG branch: 1D-ResNet (upgraded from BiLSTM in Exp 3)
- ECG auxiliary loss weight: {HP['ecg_aux_weight']}
- In Experiment 3: ECG gate/attention was near zero, Text and Image dominated.
- Best Val AUC this experiment: {max(hist_v2['val_auc']):.4f}"""

interpretation = agent_call(interpret_system, interpret_prompt, max_tokens=1000)
print(interpretation)

In [ ]:
# ====================================
# Cell 12: AI Agent — Clinical Case Study (Enhanced)
# ====================================

print("="*60)
print("🤖 AI Agent: Enhanced Clinical Case Study")
print("="*60)

class EnhancedIntentAgent:
    """AI Agent that provides structured clinical decision support.
    Integrates model predictions, attention patterns, and clinical context."""
    
    def __init__(self, client):
        self.client = client
        self.case_history = []  # Track all analyzed cases
    
    def analyze_case(self, sample_info, risk_score, gate_values, attn_matrix,
                     ecg_features=None, case_id=0):
        """Full case analysis with structured clinical output."""
        
        # Determine which modality the model focused on
        modality_names = ['Clinical Text', 'ECG Signal', 'Chest X-Ray']
        if isinstance(gate_values, (np.ndarray, list)):
            gate_arr = np.array(gate_values).flatten()[:3]
            dominant = modality_names[np.argmax(gate_arr)]
            gate_str = ', '.join([f"{m}: {g:.3f}" for m, g in zip(modality_names, gate_arr)])
        else:
            dominant = 'Unknown'
            gate_str = 'N/A'
        
        risk_level = 'HIGH' if risk_score > 0.5 else 'MODERATE' if risk_score > 0.3 else 'LOW'
        text_snippet = str(sample_info.get('text', ''))[:800]
        
        system_prompt = """You are an expert Cardiologist AI Assistant integrated into a clinical decision
support system (CDSS). Your role is to:
1. Synthesize multimodal AI predictions with clinical evidence
2. Provide actionable risk stratification
3. Recommend specific interventions based on risk level
4. Flag any concerns about the AI's attention pattern

Structure your response EXACTLY as:
## Risk Assessment
[Risk level, probability, confidence commentary]

## Key Clinical Evidence
[Bullet points of specific findings from the notes]

## AI Attention Pattern
[Which modality contributed most and clinical interpretation]

## Recommended Actions
[Specific follow-up, medication adjustments, monitoring plan]

## Flags & Limitations
[Any concerns about the prediction or data quality]"""
        
        user_prompt = f"""Case #{case_id + 1} Analysis Request

Patient Data Source: {sample_info.get('data_source', 'Unknown')}
True Label: {sample_info.get('label', 'Unknown')}

Clinical Notes Excerpt:
\"{text_snippet}\"

Model Output:
- 30-Day Readmission Risk: {risk_score:.4f} ({risk_level})
- Dominant Modality: {dominant}
- Gate Values: {gate_str}
- ECG Branch Active: {'Yes' if gate_arr[1] > 0.2 else 'Minimal'}"""
        
        report = agent_call(system_prompt, user_prompt, max_tokens=800)
        
        # Store case for summary
        self.case_history.append({
            'case_id': case_id,
            'risk_score': risk_score,
            'risk_level': risk_level,
            'dominant_modality': dominant,
            'true_label': sample_info.get('label', -1)
        })
        
        return report
    
    def generate_cohort_summary(self):
        """Generate overall summary of all analyzed cases."""
        if not self.case_history:
            return "No cases analyzed yet."
        
        summary_system = """You are a clinical AI analyst. Given a batch of case analyses,
provide a cohort-level summary including:
1. Overall risk distribution
2. Common patterns in high-risk cases
3. Model reliability assessment
4. Recommendations for clinical deployment"""
        
        summary_prompt = f"""Analyzed {len(self.case_history)} cases:
{json.dumps(self.case_history, indent=2)}"""
        
        return agent_call(summary_system, summary_prompt, max_tokens=600)


# ---- Run Enhanced Case Studies ----
agent_v2 = EnhancedIntentAgent(client)
model_v2.eval()

cases_found = 0
max_cases = 3  # Analyze 3 diverse cases

with torch.no_grad():
    for batch in test_loader:
        if cases_found >= max_cases:
            break
        
        for i in range(len(batch['label'])):
            if cases_found >= max_cases:
                break
            
            ecg = batch['ecg'][i].unsqueeze(0).to(device)
            img = batch['image'][i].unsqueeze(0).to(device)
            txt = [batch['text_raw'][i]]
            
            inp = tok_v2(txt, padding=True, truncation=True, return_tensors='pt', max_length=128)
            ids = inp['input_ids'].to(device)
            msk = inp['attention_mask'].to(device)
            
            output = model_v2(ecg, img, ids, msk)
            logits, weights, gates = output[0], output[1], output[2]
            prob = torch.sigmoid(logits).item()
            
            # Select diverse cases: high risk, low risk, borderline
            if cases_found == 0 and prob > 0.5:  # High risk
                pass
            elif cases_found == 1 and prob < 0.3:  # Low risk
                pass
            elif cases_found == 2 and 0.3 <= prob <= 0.7:  # Borderline
                pass
            else:
                continue
            
            gate_vals = gates.cpu().numpy().flatten() if gates is not None else [0.33, 0.33, 0.33]
            
            # Get attention matrix
            if weights is not None:
                if weights.dim() == 3:
                    attn_mat = weights.squeeze(0).cpu().numpy()
                elif weights.dim() == 4:
                    attn_mat = weights.mean(dim=1).squeeze(0).cpu().numpy()
                else:
                    attn_mat = np.eye(3)
            else:
                attn_mat = np.eye(3)
            
            sample_info = {
                'data_source': batch['data_source'][i],
                'text': batch['text_raw'][i],
                'label': float(batch['label'][i])
            }
            
            case_type = ['High Risk', 'Low Risk', 'Borderline'][cases_found]
            print(f"\n{'='*60}")
            print(f"📋 Case {cases_found+1}: {case_type} (Pred={prob:.4f}, True={batch['label'][i]:.0f})")
            print(f"   Gates: Text={gate_vals[0]:.3f}, ECG={gate_vals[1]:.3f}, Image={gate_vals[2]:.3f}")
            print(f"{'='*60}")
            
            report = agent_v2.analyze_case(sample_info, prob, gate_vals, attn_mat, case_id=cases_found)
            print(report)
            
            cases_found += 1

# ---- Cohort Summary ----
if cases_found > 0:
    print("\n" + "="*60)
    print("🤖 AI Agent: Cohort-Level Summary")
    print("="*60)
    cohort_summary = agent_v2.generate_cohort_summary()
    print(cohort_summary)

In [ ]:
# ====================================
# Cell 13: Ablation Study — Improvement Decomposition
# ====================================

print("="*60)
print("Ablation Study: Which improvements matter most?")
print("="*60)

def quick_evaluate(model, loader, device, tokenizer):
    """Quick evaluation returning AUC and F1."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ecg = batch['ecg'].to(device)
            img = batch['image'].to(device)
            tgt = batch['label'].to(device).unsqueeze(1)
            texts = batch['text_raw']
            inputs = tokenizer(list(texts), padding=True, truncation=True,
                             return_tensors='pt', max_length=128)
            ids = inputs['input_ids'].to(device)
            msk = inputs['attention_mask'].to(device)
            
            output = model(ecg, img, ids, msk)
            logits = output[0]
            preds.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            targets.extend(tgt.cpu().numpy().flatten())
    try:
        if len(np.unique(targets)) > 1:
            auc_val = roc_auc_score(targets, preds)
            f1 = f1_score(targets, (np.array(preds) > 0.5).astype(int))
            return auc_val, f1
    except:
        pass
    return 0.5, 0.0

# ---- Modality masking ablation on v2 model ----
print("\nModality Ablation (masking individual modalities):")

def evaluate_ablation_v2(model, loader, device, tokenizer, mask_modality=None):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ecg = batch['ecg'].to(device)
            img = batch['image'].to(device)
            tgt = batch['label'].to(device).unsqueeze(1)
            texts = batch['text_raw']
            inputs = tokenizer(list(texts), padding=True, truncation=True,
                             return_tensors='pt', max_length=128)
            ids = inputs['input_ids'].to(device)
            msk = inputs['attention_mask'].to(device)
            
            if mask_modality == 'text': msk = torch.zeros_like(msk)
            if mask_modality == 'ecg': ecg = torch.zeros_like(ecg)
            if mask_modality == 'image': img = torch.zeros_like(img)
            
            output = model(ecg, img, ids, msk)
            logits = output[0]
            preds.extend(torch.sigmoid(logits).cpu().numpy().flatten())
            targets.extend(tgt.cpu().numpy().flatten())
    try:
        auc_val = roc_auc_score(targets, preds) if len(np.unique(targets)) > 1 else 0.5
        f1 = f1_score(targets, (np.array(preds) > 0.5).astype(int))
    except:
        auc_val, f1 = 0.5, 0.0
    return auc_val, f1

full_auc, full_f1 = quick_evaluate(model_v2, test_loader, device, tok_v2)
no_text_auc, no_text_f1 = evaluate_ablation_v2(model_v2, test_loader, device, tok_v2, 'text')
no_ecg_auc, no_ecg_f1 = evaluate_ablation_v2(model_v2, test_loader, device, tok_v2, 'ecg')
no_img_auc, no_img_f1 = evaluate_ablation_v2(model_v2, test_loader, device, tok_v2, 'image')

ablation_data = {
    'Full Model': (full_auc, full_f1),
    'w/o Text': (no_text_auc, no_text_f1),
    'w/o ECG': (no_ecg_auc, no_ecg_f1),
    'w/o Image': (no_img_auc, no_img_f1)
}

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC ablation
ax = axes[0]
labels = list(ablation_data.keys())
auc_vals = [v[0] for v in ablation_data.values()]
colors_ab = ['#E53935', '#90CAF9', '#A5D6A7', '#FFCC80']
bars = ax.bar(labels, auc_vals, color=colors_ab, edgecolor='gray')
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Modality Ablation — AUC Impact', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold')
    # Show delta
    if val != full_auc:
        delta = val - full_auc
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() - 0.06,
                f'Δ={delta:+.3f}', ha='center', fontsize=9, color='red' if delta < 0 else 'green')

# F1 ablation
ax = axes[1]
f1_vals = [v[1] for v in ablation_data.values()]
bars = ax.bar(labels, f1_vals, color=colors_ab, edgecolor='gray')
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title('Modality Ablation — F1 Impact', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('Experiment 4: Modality Contribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'fig_exp4_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

# Print
print(f"\n{'Condition':20s} {'AUC':>8s} {'F1':>8s} {'ΔAUC':>8s}")
print("-"*48)
for name, (a, f) in ablation_data.items():
    delta = a - full_auc
    print(f"{name:20s} {a:8.3f} {f:8.3f} {delta:+8.3f}")

In [ ]:
# ====================================
# Cell 14: ECG Trajectory with Model Attention
# ====================================

model_v2.eval()
found = False

with torch.no_grad():
    for batch in test_loader:
        if found:
            break
        for i in range(len(batch['label'])):
            if batch['label'][i] == 1 and batch['has_ecg'][i]:
                ecg_np = batch['ecg'][i].flatten().numpy()
                txt_preview = batch['text_raw'][i][:80] + "..."
                
                # Get model prediction for this sample
                ecg_single = batch['ecg'][i].unsqueeze(0).to(device)
                img_single = batch['image'][i].unsqueeze(0).to(device)
                inp = tok_v2([batch['text_raw'][i]], padding=True, truncation=True,
                           return_tensors='pt', max_length=128)
                ids = inp['input_ids'].to(device)
                msk = inp['attention_mask'].to(device)
                
                output = model_v2(ecg_single, img_single, ids, msk)
                logits, weights, gates = output[0], output[1], output[2]
                prob = torch.sigmoid(logits).item()
                gate_vals = gates.cpu().numpy().flatten() if gates is not None else [0.33, 0.33, 0.33]
                
                fig, axes = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]})
                
                # ECG signal
                ax = axes[0]
                pts = min(1500, len(ecg_np))
                ax.plot(ecg_np[:pts], color='black', lw=0.7)
                ax.axvspan(int(pts*0.15), int(pts*0.3), color='red', alpha=0.15, label='Elevated Risk Region')
                ax.axvspan(int(pts*0.6), int(pts*0.75), color='red', alpha=0.15)
                ax.set_title(f'Patient ECG Trajectory — High Risk (Pred={prob:.3f})\n'
                            f'Clinical Context: {txt_preview}', fontsize=11)
                ax.set_xlabel('Time Steps')
                ax.set_ylabel('Normalized Amplitude')
                ax.legend(loc='upper right')
                
                # Gate values bar
                ax2 = axes[1]
                bar_colors = ['#1E88E5', '#E53935', '#43A047']
                bars = ax2.barh(['Text', 'ECG', 'Image'], gate_vals, color=bar_colors, height=0.5)
                ax2.set_xlim(0, 1)
                ax2.set_xlabel('Gate Value (Model Attention Weight)')
                ax2.set_title('Modality Gate Values for This Patient', fontsize=11)
                for bar, val in zip(bars, gate_vals):
                    ax2.text(val + 0.02, bar.get_y() + bar.get_height()/2,
                            f'{val:.3f}', va='center', fontweight='bold')
                
                plt.tight_layout()
                plt.savefig(os.path.join(BASE, 'fig_exp4_ecg_trajectory.png'), dpi=150, bbox_inches='tight')
                plt.show()
                found = True
                break

if not found:
    print("No high-risk ECG case found for trajectory visualization.")

In [ ]:
# ====================================
# Cell 15: AI Agent — Final Experiment Summary & Recommendations
# ====================================

print("="*60)
print("🤖 AI Agent: Experiment 4 Final Summary Report")
print("="*60)

# Compile all experiment results
final_metrics = {
    'HyCF-TFM_v2': {
        'best_auc': max(hist_v2['val_auc']),
        'best_f1': max(hist_v2['val_f1']),
        'best_ap': max(hist_v2['val_ap']),
        'best_recall': max(hist_v2['val_recall']),
        'best_precision': max(hist_v2['val_precision']),
        'epochs_trained': len(hist_v2['train_loss']),
        'final_gate_values': hist_v2['gate_values'][-1] if hist_v2['gate_values'] else None,
        'final_contrastive_loss': hist_v2['contrastive_loss'][-1] if hist_v2.get('contrastive_loss') else None,
        'final_gate_entropy': hist_v2['gate_entropy'][-1] if hist_v2.get('gate_entropy') else None,
        'num_agent_interventions': len(hist_v2.get('agent_interventions', []))
    },
    'Concat_v2': {
        'best_auc': max(hist_concat['val_auc']),
        'best_f1': max(hist_concat['val_f1']),
        'best_ap': max(hist_concat['val_ap']),
        'best_recall': max(hist_concat['val_recall']),
        'best_precision': max(hist_concat['val_precision']),
        'epochs_trained': len(hist_concat['train_loss'])
    }
}

improvements_applied = [
    'WeightedRandomSampler (1:1 class balance)',
    f'Focal Loss (gamma={HP["focal_gamma"]}, alpha={HP["focal_alpha"]})',
    f'Label Smoothing ({HP["label_smoothing"]})',
    '1D-ResNet ECG branch (replacing BiLSTM)',
    f'ECG auxiliary loss (weight={HP["ecg_aux_weight"]})',
    'Softmax Gated Cross-Modal Fusion (2 layers) + Entropy Regularization',
    f'Contrastive Alignment Loss (InfoNCE, temp={HP["contrastive_temperature"]}, weight={HP["contrastive_weight"]})',
    f'Text Feature Masking (prob={HP["text_mask_prob"]}) — break text hegemony',
    f'Staged Training: first {HP["staged_epochs"]} epochs ECG+Image only',
    f'Asymmetric ECG LR (multiplier={HP["lr_ecg_multiplier"]}x)',
    f'Increased Dropout ({HP["dropout"]})',
    f'Differential LR (BERT={HP["lr_bert"]}, ECG={HP["lr_other"]*HP["lr_ecg_multiplier"]}, Other={HP["lr_other"]})',
    'CosineAnnealing LR scheduler',
    f'Agent Closed-Loop Intervention (every {HP["agent_intervene_every"]} epochs)',
    f'Agent-Generated Intent Attention Vector (clinical prior initialization)',
    f'Gradient clipping ({HP["grad_clip"]})',
    f'Early stopping (patience={HP["patience"]})',
    'Positive sample augmentation (ECG noise/shift, Image flip/jitter)'
]

summary_system = """You are a senior AI researcher writing the conclusion of a medical AI experiment.
This experiment implemented an "Intent-Driven" multimodal fusion framework with several key innovations:
1. Contrastive Alignment Loss (CLIP-style InfoNCE) for true cross-modal alignment
2. Softmax gates with entropy regularization for balanced modality competition
3. Staged training (ECG+Image first) to prevent text hegemony
4. Agent closed-loop intervention for dynamic hyperparameter tuning
5. Intent attention vectors from clinical prior knowledge

Provide a comprehensive summary covering:
1. **Experiment Overview**: The "Intent-Driven" methodology contribution
2. **Key Results**: Quantitative improvements with specific numbers
3. **Contrastive Alignment Effectiveness**: Did InfoNCE loss improve cross-modal alignment?
4. **Gate Balance**: Did Softmax gates + entropy regularization balance modalities?
5. **Staged Training Impact**: Did ECG get more attention in the final model?
6. **Agent Closed-Loop Value**: Did the Agent interventions demonstrably help?
7. **Clinical Relevance**: How these innovations translate to clinical utility
8. **Next Steps**: Concrete recommendations for Experiment 5

Be specific, quantitative, and scientifically rigorous. Reference the actual numbers."""

summary_prompt = f"""Experiment 4 Results — Intent-Driven Multimodal Fusion:

Metrics:
{json.dumps(final_metrics, indent=2)}

Improvements Applied:
{chr(10).join(f'  - {x}' for x in improvements_applied)}

Contrastive Loss Trajectory: {hist_v2.get('contrastive_loss', [])}
Gate Entropy Trajectory: {hist_v2.get('gate_entropy', [])}
Training Phases: {hist_v2.get('training_phase', [])}
Agent Interventions: {json.dumps(hist_v2.get('agent_interventions', []))}

Ablation Results:
  Full Model AUC: {full_auc:.4f}
  w/o Text AUC:   {no_text_auc:.4f} (Δ={no_text_auc-full_auc:+.4f})
  w/o ECG AUC:    {no_ecg_auc:.4f} (Δ={no_ecg_auc-full_auc:+.4f})
  w/o Image AUC:  {no_img_auc:.4f} (Δ={no_img_auc-full_auc:+.4f})

Dataset: {len(train_dataset)} train, {len(test_dataset)} test
Class balance: {n_pos_train} pos / {n_neg_train} neg (originally {pos_rate:.1f}% positive)

Context: In Experiment 3, the model had:
- All models predicted all-negative (F1=0)
- ECG attention was near zero
- Validation loss diverged at epoch 3
- HyCF-TFM did not outperform Concat baseline"""

final_summary = agent_call(summary_system, summary_prompt, max_tokens=1500)
print(final_summary)

print("\n" + "="*60)
print("Experiment 4 Complete — Intent-Driven Multimodal Fusion")
print("="*60)
print(f"Saved figures:")
print(f"  fig_exp4_comparison.png          — ROC/PR/Loss/Gates comparison")
print(f"  fig_exp4_alignment_diagnostics.png — Contrastive loss + Gate entropy + Agent timeline")
print(f"  fig_exp4_attention.png           — Cross-modal attention heatmap")
print(f"  fig_exp4_ablation.png            — Modality ablation study")
print(f"  fig_exp4_ecg_trajectory.png      — ECG signal with gate values")
print(f"Saved model: hycf_tfm_v2_model.pth")
print(f"\nKey Innovations Implemented:")
print(f"  1. Contrastive Alignment Loss (InfoNCE)")
print(f"  2. Softmax Gates + Entropy Regularization")
print(f"  3. Staged Training (ECG+Image first {HP['staged_epochs']} epochs)")
print(f"  4. Agent Closed-Loop Intervention ({len(hist_v2.get('agent_interventions', []))} interventions)")
print(f"  5. Intent Attention Vector (clinical prior initialization)")
print(f"  6. Text Feature Masking ({HP['text_mask_prob']*100:.0f}%)")
print("="*60)